<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/05-Improve_Prompts_+_Add_Source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.0 openai==1.107.0 chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.2 \
                llama-index-llms-google-genai==0.3.0 jedi==0.19.2

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    pass  # Running locally — keys are expected in the environment

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# Create a VectoreStore


In [ ]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

## Load the Articles


In [ ]:
import csv

rows = []

# Load the file as a JSON
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

# The number of characters in the dataset.
len(rows)

# Convert to Document obj


In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1], metadata={"title": row[0], "url": row[2], "source_name": row[3]}
    )
    for row in rows
]

In [ ]:
len(documents)

In [ ]:
print(documents[0].text)
documents[0].metadata

# Transforming


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter

# Define the splitter object that split the text into segments with 512 tokens,
# with a 128 overlap between the segments.
text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

# Create the pipeline to apply the transformation (splitting and embedding) on each chunk,
# and store the transformed text in the chroma vector store.
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(model = 'text-embedding-3-small'),
    ],
    vector_store=vector_store,
)

# Run the transformation pipeline.
b = pipeline.run(documents=documents, show_progress=True)

# Load Indexes


In [ ]:
# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = db.get_or_create_collection("mini-llama-articles")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index based on the vector store.
index = VectorStoreIndex.from_vector_store(vector_store)

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    generation_config=config,
    )

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
res = query_engine.query("How many parameters LLaMA2 model has?")

In [ ]:
res.response

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

# Response Modes


The behavior of the query engine during response generation can be adjusted. Several modes are available for consideration, including the following:

- compact (default): Concatenate all the retrieved chunks and use them in the prompt to generate an answer.
- refine: Generate an answer based on the first retrieved chunk, then improve the answer based on the other retrieved chunks one at a time. (will send one request for each chunk to refine the response)
- tree summarize: concatenate the retrieved chunks until they fit the context window and summarize them. The summaized chunks will then recusively fed back to the LLM for summarization until one chunk remains which would be the final answer.

Refer to [documentation](https://docs.llamaindex.ai/en/stable/module_guides/querying/response_synthesizers/root.html#configuring-the-response-mode) for a comprehensive list.

Due to the limited size of the sample dataset, the examples provided will yield identical responses. It's crucial to evaluate these methods in the context of your specific use case and cost considerations.


In [ ]:
query_engine = index.as_query_engine(response_mode="refine", llm=llm)
#query_engine = index.as_query_engine(response_mode="tree_summarize")

In [ ]:
res = query_engine.query("How many parameters LLaMA 2 model has?")

In [ ]:
res.response

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

The `no_text` mode will retrieve the documents, but will not send the request to the API to synthesize the final response. It is a great approach to debug the retrieved documents.


In [ ]:
query_engine = index.as_query_engine(response_mode="no_text", llm=llm)
res = query_engine.query("How many parameters LLaMA 2 model has?")

In [ ]:
res.response

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

## Optional: Custom Prompt Template

Customize the LlamaIndex prompt template to enforce specific answer formatting,
such as always including a confidence disclaimer or a structured response.

In [ ]:
from llama_index.core import PromptTemplate

# Custom QA prompt that asks the model to cite the source title
custom_qa_tmpl_str = (
    "You are an expert AI assistant. Use only the context below to answer the question.\n"
    "Always end your answer with: 'Source: {source_title}'\n\n"
    "Context:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Question: {query_str}\n"
    "Answer: "
)

# We use compact mode (default) with a custom template
query_engine_custom = index.as_query_engine(
    llm=llm,
    similarity_top_k=3,
    response_mode="compact",
)

# Update the QA template
custom_qa_tmpl = PromptTemplate(custom_qa_tmpl_str)
query_engine_custom.update_prompts(
    {"response_synthesizer:text_qa_template": custom_qa_tmpl}
)

custom_res = query_engine_custom.query("What is Ghost Attention in LLaMA 2?")
print("Custom prompt response:")
print(custom_res.response)

## Optional: Response Modes Side-by-Side Comparison

Compare `compact`, `refine`, and `tree_summarize` response modes on the same query
so you can see how each approach affects the output quality and length.

In [ ]:
compare_q = "Summarize the key innovations in LLaMA 2."

for mode in ["compact", "refine", "tree_summarize"]:
    qe = index.as_query_engine(response_mode=mode, llm=llm, similarity_top_k=3)
    r = qe.query(compare_q)
    print(f"=== Mode: {mode} ===")
    print(str(r.response)[:400])
    print()